# 11 — KalmanNet Adaptive Filtering Training

**SIH PS 26168 — Intelligent Dead Reckoning**

> **Roadmap Section 19:** Use KalmanNet-style learning for adaptive filtering.
> Adapt covariance, measurement confidence, and dynamic filtering gain.
> Strict prohibition of arbitrary fixed gains ($K = 0.80$).

### Pipeline Overview:
1. **State-Space Formulation**: State $\mathbf{x} = [p_e, p_n, v_e, v_n]^T$ (4-DOF), Observation $\mathbf{z} = [v_e, v_n]^T$ (2-DOF).
2. **Recurrent Architecture**: 2-layer GRU (64 units) mapping filter innovations & residuals to Kalman Gain $\mathbf{K}_k$.
3. **Physics-Constrained Propagation**: Explicit kinematic motion model with learned innovation injection.
4. **Training**: Backpropagation Through Time (BPTT) on trajectory sequences using real IO-VNBD dataset.

## 1. Environment & Hardware Setup

In [ ]:
import os, sys, time, json
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.kalmannet import KalmanNetNN
from src.datasets.kalmannet_dataset import KalmanNetDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Compute Device: {device}')
if torch.cuda.is_available():
    print(f'GPU Hardware: {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')

ckpt_dir = PROJECT_ROOT / 'checkpoints' / 'kalmannet'
plots_dir = PROJECT_ROOT / 'plots' / 'kalmannet'
ckpt_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

## 2. Load Trajectory Datasets

In [ ]:
io_ckpt = PROJECT_ROOT / 'checkpoints' / 'inertial_odometry' / 'inertial_odometry_best.pt'
io_ckpt_str = str(io_ckpt) if io_ckpt.exists() else None

print('Initializing Train & Validation Datasets for KalmanNet...')
train_ds = KalmanNetDataset(
    split='train',
    seq_len=50,
    stride=20,
    io_checkpoint_path=io_ckpt_str
)

val_ds = KalmanNetDataset(
    split='val',
    seq_len=50,
    stride=25,
    io_checkpoint_path=io_ckpt_str
)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train Sequences: {len(train_ds)} ({len(train_loader)} batches)')
print(f'Val Sequences  : {len(val_ds)} ({len(val_loader)} batches)')

## 3. Instantiate KalmanNet Model & Physics Matrices

In [ ]:
model = KalmanNetNN(
    state_dim=4,
    meas_dim=2,
    hidden_dim=64,
    num_layers=2,
    dropout=0.1
).to(device)

param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'KalmanNet Initialized. Trainable Parameters: {param_count:,}')

dt = 0.1
F_mat = torch.tensor([
    [1.0, 0.0, dt,  0.0],
    [0.0, 1.0, 0.0, dt ],
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 1.0]
], dtype=torch.float32, device=device)

B_mat = torch.tensor([
    [0.5 * dt**2, 0.0],
    [0.0, 0.5 * dt**2],
    [dt, 0.0],
    [0.0, dt]
], dtype=torch.float32, device=device)

H_mat = torch.tensor([
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 1.0]
], dtype=torch.float32, device=device)

## 4. End-to-End Sequence Step Function

In [ ]:
def rollout_kalmannet(model, a_nav, z_meas, init_state, F_m, B_m, H_m):
    """
    Differentiable rollout of KalmanNet over a sequence batch.
    """
    B, L, _ = a_nav.shape
    x_curr = init_state
    z_prev = z_meas[:, 0, :]
    h_prev = None

    post_states = []
    gains = []

    for t in range(1, L):
        a_t = a_nav[:, t, :]
        z_t = z_meas[:, t, :]

        # Physics Prior Prediction
        x_prior = torch.matmul(x_curr, F_m.T) + torch.matmul(a_t, B_m.T)

        # KalmanNet Adaptive Filter Step
        x_post, K_gain, h_prev = model.step(
            x_prior=x_prior,
            z_meas=z_t,
            H_matrix=H_m,
            x_prev=x_curr,
            z_prev=z_prev,
            h_prev=h_prev
        )

        post_states.append(x_post)
        gains.append(K_gain)
        x_curr = x_post
        z_prev = z_t

    pred_states = torch.stack(post_states, dim=1)  # (B, L-1, 4)
    stack_gains = torch.stack(gains, dim=1)        # (B, L-1, 4, 2)
    return pred_states, stack_gains

print('Rollout function compiled successfully.')

## 5. Training Loop with BPTT

In [ ]:
EPOCHS = 50
LEARNING_RATE = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

best_val_loss = float('inf')
best_ckpt_path = ckpt_dir / 'kalmannet_best.pt'

history = {
    'train_loss': [],
    'val_loss': [],
    'train_pos_rmse': [],
    'val_pos_rmse': [],
    'train_vel_rmse': [],
    'val_vel_rmse': []
}

print(f'Starting KalmanNet Training for {EPOCHS} epochs on {device}...')
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    t_losses, t_pos_errs, t_vel_errs = [], [], []

    for batch in train_loader:
        a_nav = batch['a_nav'].to(device)
        z_meas = batch['z_meas'].to(device)
        gt_state = batch['gt_state'].to(device)
        init_state = batch['init_state'].to(device)

        optimizer.zero_grad()
        pred_states, _ = rollout_kalmannet(model, a_nav, z_meas, init_state, F_mat, B_mat, H_mat)
        gt_target = gt_state[:, 1:, :]

        # Loss computation: Position MSE + Velocity MSE
        pos_diff = pred_states[:, :, 0:2] - gt_target[:, :, 0:2]
        vel_diff = pred_states[:, :, 2:4] - gt_target[:, :, 2:4]

        loss_pos = torch.mean(pos_diff ** 2)
        loss_vel = torch.mean(vel_diff ** 2)
        loss = loss_pos + 0.5 * loss_vel

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        t_losses.append(loss.item())
        t_pos_errs.append(torch.sqrt(loss_pos).item())
        t_vel_errs.append(torch.sqrt(loss_vel).item())

    scheduler.step()

    # Validation Loop
    model.eval()
    v_losses, v_pos_errs, v_vel_errs = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            a_nav = batch['a_nav'].to(device)
            z_meas = batch['z_meas'].to(device)
            gt_state = batch['gt_state'].to(device)
            init_state = batch['init_state'].to(device)

            pred_states, _ = rollout_kalmannet(model, a_nav, z_meas, init_state, F_mat, B_mat, H_mat)
            gt_target = gt_state[:, 1:, :]

            pos_diff = pred_states[:, :, 0:2] - gt_target[:, :, 0:2]
            vel_diff = pred_states[:, :, 2:4] - gt_target[:, :, 2:4]

            loss_pos = torch.mean(pos_diff ** 2)
            loss_vel = torch.mean(vel_diff ** 2)
            loss = loss_pos + 0.5 * loss_vel

            v_losses.append(loss.item())
            v_pos_errs.append(torch.sqrt(loss_pos).item())
            v_vel_errs.append(torch.sqrt(loss_vel).item())

    mean_t_loss = np.mean(t_losses)
    mean_v_loss = np.mean(v_losses)
    mean_t_pos = np.mean(t_pos_errs)
    mean_v_pos = np.mean(v_pos_errs)
    mean_t_vel = np.mean(t_vel_errs)
    mean_v_vel = np.mean(v_vel_errs)

    history['train_loss'].append(mean_t_loss)
    history['val_loss'].append(mean_v_loss)
    history['train_pos_rmse'].append(mean_t_pos)
    history['val_pos_rmse'].append(mean_v_pos)
    history['train_vel_rmse'].append(mean_t_vel)
    history['val_vel_rmse'].append(mean_v_vel)

    if mean_v_loss < best_val_loss:
        best_val_loss = mean_v_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
            'val_pos_rmse': mean_v_pos,
            'val_vel_rmse': mean_v_vel,
            'config': {
                'state_dim': 4,
                'meas_dim': 2,
                'hidden_dim': 64,
                'num_layers': 2
            }
        }, best_ckpt_path)
        star = ' *** (Best Checkpoint Saved)'
    else:
        star = ''

    if epoch % 5 == 0 or epoch == 1 or star != '':
        print(f'Epoch {epoch:02d}/{EPOCHS} | Train Loss: {mean_t_loss:.4f} (Pos: {mean_t_pos:.2f}m, Vel: {mean_t_vel:.2f}m/s) | '
              f'Val Loss: {mean_v_loss:.4f} (Pos: {mean_v_pos:.2f}m, Vel: {mean_v_vel:.2f}m/s){star}')

elapsed = time.time() - start_time
print(f'Training Completed in {elapsed:.1f}s. Best Val Loss: {best_val_loss:.4f}')

## 6. Plot & Save Training Diagnostics

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss (Pos+0.5*Vel)', color='#1f77b4', lw=2)
plt.plot(history['val_loss'], label='Val Loss', color='#ff7f0e', lw=2)
plt.xlabel('Epoch')
plt.ylabel('Trajectory Loss')
plt.title('KalmanNet Training Loss Curve')
plt.grid(True, alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_pos_rmse'], label='Train Pos RMSE (m)', color='#2ca02c', lw=1.8)
plt.plot(history['val_pos_rmse'], label='Val Pos RMSE (m)', color='#d62728', lw=1.8)
plt.plot(history['train_vel_rmse'], label='Train Vel RMSE (m/s)', color='#9467bd', lw=1.5, ls='--')
plt.plot(history['val_vel_rmse'], label='Val Vel RMSE (m/s)', color='#8c564b', lw=1.5, ls='--')
plt.xlabel('Epoch')
plt.ylabel('RMSE')
plt.title('KalmanNet State Estimation Errors')
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
curve_path = plots_dir / 'kalmannet_training_curve.png'
plt.savefig(curve_path, dpi=150)
plt.close()
print(f'Training curve saved to: {curve_path}')

metrics_path = PROJECT_ROOT / 'results' / 'kalmannet_training_metrics.json'
metrics_path.parent.mkdir(parents=True, exist_ok=True)
with open(metrics_path, 'w') as f:
    json.dump({
        'best_val_loss': best_val_loss,
        'best_epoch': epoch,
        'final_train_pos_rmse': history['train_pos_rmse'][-1],
        'final_val_pos_rmse': history['val_pos_rmse'][-1],
        'final_train_vel_rmse': history['train_vel_rmse'][-1],
        'final_val_vel_rmse': history['val_vel_rmse'][-1]
    }, f, indent=2)
print(f'Metrics exported to: {metrics_path}')